# 󠁧󠁢󠁥󠁮󠁧󠁿 Web Scraping Football Matches From The EPL in Python

This notebook scrapes English Premier League (EPL) match data from [FBref](https://fbref.com) for multiple seasons and teams, then saves the cleaned data to a CSV file ready for machine learning.

**Data Source:** [FBref — Premier League Stats](https://fbref.com/en/comps/9/Premier-League-Stats)

---

## Step 1: Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print(" Libraries imported successfully!")

## Step 2: Download the EPL Stats Page (2021–2022)

We start with the [2021–2022 EPL season stats page](https://fbref.com/en/comps/9/2021-2022/2021-2022-Premier-League-Stats) to get the **League Table** and extract each team's URL.

In [ ]:
# Define the target URL — 2021-2022 EPL season standings
url = "https://fbref.com/en/comps/9/2021-2022/2021-2022-Premier-League-Stats"

# Download the HTML page
response = requests.get(url)
print(f"Status Code : {response.status_code}")
print(f"Content size: {len(response.text):,} characters")

## Step 3: Parse HTML Links with Beautiful Soup

Using Beautiful Soup, we scope to the first table (**League Table**) and extract the absolute URLs for all 20 EPL teams.

In [ ]:
# Parse the page; scope to the first stats table
soup = BeautifulSoup(response.text, "html.parser")
standings_table = soup.select("table.stats_table")[0]

# Extract all anchor tags and grab href attributes
links = standings_table.find_all("a")
links = [link.get("href") for link in links]

# Keep only links that point to squad/team pages
links = [link for link in links if link and "/squads/" in link]

# Build full absolute URLs
team_urls = [f"https://fbref.com{link}" for link in links]

print(f"Found {len(team_urls)} team URLs:")
for t in team_urls:
    print(" •", t)

## Step 4: Extract Match Stats — Manchester City

Let's start with the first team and read the **Scores & Fixtures** table into a pandas DataFrame.

In [ ]:
# Select the first team URL
team_url = team_urls[0]
print(f"Fetching: {team_url}")

# Download the team's page
team_data = requests.get(team_url)
print(f"Status Code: {team_data.status_code}")

In [ ]:
# Read the 'Scores & Fixtures' table into a DataFrame
matches = pd.read_html(team_data.text, match="Scores & Fixtures")[0]
print(f"Shape: {matches.shape}")
matches.head()

## Step 5: Get Match Shooting Stats

The **Scores & Fixtures** table lacks shooting details. We find the team's shooting stats page (under the *Shooting* tab) and read it in separately.

In [ ]:
# Parse the team page to find the shooting stats link
team_soup = BeautifulSoup(team_data.text, "html.parser")
all_links = [a.get("href") for a in team_soup.find_all("a")]

# Filter for shooting stats links
shooting_links = [l for l in all_links if l and "shooting" in l]

# Build full URL for shooting stats
shooting_url = f"https://fbref.com{shooting_links[0]}"
print(f"Shooting stats URL:\n  {shooting_url}")

In [ ]:
# Download the shooting stats page
shooting_data = requests.get(shooting_url)
print(f"Status Code: {shooting_data.status_code}")

# Read the Shooting table into a DataFrame
shooting = pd.read_html(shooting_data.text, match="Shooting")[0]
print(f"Shape: {shooting.shape}")
shooting.head()

## Step 6: Clean and Merge the DataFrames

The shooting DataFrame has a **multi-level column index**. We drop the top level, then merge both DataFrames on the `Date` column.

In [ ]:
# Inspect the multi-level index
print("Raw columns (first 10):")
print(shooting.columns[:10].tolist())

In [ ]:
# Drop the top-level of the MultiIndex to flatten column names
shooting.columns = shooting.columns.droplevel()
print("Flattened columns:")
print(shooting.columns.tolist())

In [ ]:
# Merge matches with the most useful shooting columns
shooting_cols = ["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]

team_df = matches.merge(shooting[shooting_cols], on="Date")

print(f"Merged shape: {team_df.shape}")
team_df.head()

## Step 7: Scrape All Teams Across Multiple Seasons

We loop through **3 seasons** (2022–23, 2021–22, 2020–21) and all 20 teams each season, adding `team` and `season` columns, and sleeping between requests to be polite to the server.

In [ ]:
# Seasons to scrape — in reverse chronological order
years = [2022, 2021, 2020]

# Starting URL — latest EPL standings
standing_url = "https://fbref.com/en/comps/9/Premier-League-Stats"

# Accumulator
all_matches = []

print(f"Scraping seasons: {years}")

In [ ]:
for year in years:
    print(f"\n{'='*60}")
    print(f" Season ending: {year}")
    print(f"{'='*60}")

    #  Download standings page 
    response = requests.get(standing_url)
    soup = BeautifulSoup(response.text, "html.parser")

    #  Extract team URLs from first table 
    standings_table = soup.select("table.stats_table")[0]
    links = [a.get("href") for a in standings_table.find_all("a")]
    team_urls = [f"https://fbref.com{l}" for l in links if l and "/squads/" in l]
    print(f"  Teams found: {len(team_urls)}")

    #  Navigate to previous season for next loop iteration 
    prev_season_link = soup.select("a.prev")[0].get("href")
    standing_url = f"https://fbref.com{prev_season_link}"
    print(f"  Next season URL: {standing_url}")

    #  Loop through each team 
    for team_url in team_urls:
        # Derive a readable team name from the URL
        team_name = team_url.split("/")[-1].replace("-Stats", "").replace("-", " ")
        print(f"\n  → {team_name}")

        try:
            # Download team page
            team_resp = requests.get(team_url)

            # Read Scores & Fixtures table
            matches = pd.read_html(team_resp.text, match="Scores & Fixtures")[0]

            # Find shooting stats link
            t_soup = BeautifulSoup(team_resp.text, "html.parser")
            t_links = [a.get("href") for a in t_soup.find_all("a")]
            shoot_links = [l for l in t_links if l and "shooting" in l]

            if not shoot_links:
                print(f"      No shooting link — skipping")
                continue

            # Download & read shooting stats
            shoot_resp = requests.get(f"https://fbref.com{shoot_links[0]}")
            shooting = pd.read_html(shoot_resp.text, match="Shooting")[0]
            shooting.columns = shooting.columns.droplevel()   # flatten MultiIndex

            # Merge
            team_df = matches.merge(
                shooting[["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]],
                on="Date"
            )

            # Drop rows with missing shooting data
            team_df = team_df.dropna(subset=["Sh", "SoT"])

            # Keep only Premier League matches
            team_df = team_df[team_df["Comp"] == "Premier League"]

            if team_df.empty:
                print(f"      No EPL rows after filter — skipping")
                continue

            # Tag with team and season
            team_df["team"]   = team_name
            team_df["season"] = year

            all_matches.append(team_df)
            print(f"      {len(team_df)} rows added")

        except Exception as e:
            print(f"      Error — {e}")

        # Polite delay so we don't hammer the server
        time.sleep(3)

print("\n Scraping complete!")

## Step 8: Combine & Save to CSV

In [ ]:
# Concatenate all individual DataFrames
match_df = pd.concat(all_matches, ignore_index=True)

print(f"Total rows    : {match_df.shape[0]:,}")
print(f"Total columns : {match_df.shape[1]}")
print(f"Seasons       : {sorted(match_df['season'].unique())}")
print(f"Teams         : {match_df['team'].nunique()}")

match_df.head()

In [ ]:
# Save to CSV
match_df.to_csv("matches.csv", index=False)
print(" Saved to matches.csv")

In [ ]:
# Quick dataset overview
print("=== Column Types ===")
print(match_df.dtypes)
print("\n=== Missing Values ===")
print(match_df.isnull().sum())
print("\n=== Descriptive Stats ===")
match_df[["Sh", "SoT", "Dist", "FK", "PK", "PKatt"]].describe()

---
## Conclusion

We successfully scraped EPL match data including:

| Feature | Details |
|---|---|
| **Seasons** | 2020–21, 2021–22, 2022–23 |
| **Teams** | All 20 EPL clubs per season |
| **Match data** | Scores & Fixtures (date, opponent, result, venue…) |
| **Shooting stats** | Shots, Shots on Target, Distance, FK, PK, PKatt |
| **Output** | `matches.csv` |

### Next Steps
- Feature engineering (rolling averages, home/away splits, form)
- Train a machine learning model to predict match outcomes
- Evaluate and tune model performance

**Solution Reference:** [dataquestio/project-walkthroughs](https://github.com/dataquestio/project-walkthroughs/blob/master/football_matches/scraping.ipynb)